<a href="https://colab.research.google.com/github/Shashank-2004/Algorithms-and-Problem-Solving/blob/main/Lab_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Warshall algorithm to find reachability between all pairs of vertices in a graph.
Problem Description:
Understand transitive closure of a graph. Use Python to compute reachability using matrix method.

#Theory
Warshall algorithm finds whether a path exists between every pair of vertices. It updates the adjacency matrix step by step. If there is a path from i to j through k, then the entry becomes 1.

## Algorithm
1. Represent graph using adjacency matrix.
2. For each intermediate vertex k
3. Check all pairs i and j
4. Update value if path exists through k
4. Repeat for all vertices


In [ ]:
import copy

def get_matrix_from_user(n):
    print(f"Enter the {n}x{n} adjacency matrix row by row (space-separated 0s and 1s):")
    matrix = []
    for i in range(n):
        while True:
            row = input(f"Row {i+1}: ").strip().split()
            if len(row) == n and all(x in ('0', '1') for x in row):
                matrix.append([int(x) for x in row])
                break
            print(f"Invalid input. Enter exactly {n} values (0 or 1).")
    return matrix

def warshall(graph):
    g = copy.deepcopy(graph)
    n = len(g)
    for k in range(n):
        for i in range(n):
            for j in range(n):
                g[i][j] = g[i][j] or (g[i][k] and g[k][j])
    return g

def floyd(graph):
    INF = float('inf')
    n = len(graph)
    dist = [[INF] * n for _ in range(n)]
    for i in range(n):
        dist[i][i] = 0
        for j in range(n):
            if graph[i][j] == 1:
                dist[i][j] = 1
    for k in range(n):
        for i in range(n):
            for j in range(n):
                if dist[i][k] + dist[k][j] < dist[i][j]:
                    dist[i][j] = dist[i][k] + dist[k][j]
    reachable = [[1 if dist[i][j] < INF else 0 for j in range(n)] for i in range(n)]
    return reachable

def print_matrix(matrix, label):
    print(f"\n{label}:")
    for row in matrix:
        print(row)

def print_readable(matrix, n):
    print("\nPath existence (readable format):")
    for i in range(n):
        for j in range(n):
            if i != j:
                status = "reachable" if matrix[i][j] else "no path"
                print(f"  Node {i+1} -> Node {j+1}: {status}")

def compare(w_matrix, f_matrix, n):
    print("\nComparison — Warshall vs Floyd:")
    match = all(w_matrix[i][j] == f_matrix[i][j] for i in range(n) for j in range(n))
    if match:
        print("  Both algorithms produce identical results.")
    else:
        print("  Differences found:")
        for i in range(n):
            for j in range(n):
                if w_matrix[i][j] != f_matrix[i][j]:
                    print(f"    Node {i+1} -> Node {j+1}: Warshall={w_matrix[i][j]}, Floyd={f_matrix[i][j]}")

def main():
    n = 5
    graph = get_matrix_from_user(n)

    print_matrix(graph, "Input Matrix")

    w_result = warshall(graph)
    f_result = floyd(graph)

    print_matrix(w_result, "Warshall — Transitive Closure Matrix")
    print_matrix(f_result, "Floyd — Reachability Matrix")

    compare(w_result, f_result, n)
    print_readable(w_result, n)

if __name__ == "__main__":
    main()

Enter the 5x5 adjacency matrix row by row (space-separated 0s and 1s):


In [ ]:
graph = [
[0, 1, 0, 0],
[0, 0, 1, 0],
[0, 0, 0, 1],
[0, 0, 0, 0]
]

result = warshall(graph)

print("Transitive Closure Matrix:")

for row in result:
    print(row)

Transitive Closure Matrix:
[0, 1, 1, 1]
[0, 0, 1, 1]
[0, 0, 0, 1]
[0, 0, 0, 0]


Now, let's implement
1. Modify program to take matrix input from user
2. Apply algorithm on graph with 5 vertices
3. Compare Warshall with Floyd algorithm
4. Display path existence in readable format

## Find shortest paths between all pairs of vertices using Floyd Warshall algorithm.

#Understand all pairs shortest path problem. Implement dynamic programming approach in Python.

* Given a weighted graph with positive or negative edge weights, find
shortest distances between every pair of vertices.

# Theory
Floyd Warshall algorithm computes shortest paths between all pairs of nodes. It updates the distance matrix by checking if a shorter path exists through an intermediate vertex. Works with negative weights but fails with negative cycles.

Algorithm
1. Represent graph using distance matrix.
2. Initialize matrix with given weights.
3. Set distance of node to itself as 0.
4. For each intermediate vertex k
5. Update distance[i][j] = min(distance[i][j], distance[i][k] + distance[k][j])
6. Repeat for all vertices

In [ ]:
import copy
import heapq

INF = float('inf')

def get_matrix_from_user(n):
    print(f"Enter the {n}x{n} weighted adjacency matrix row by row.")
    print("Use 0 for no edge (diagonal), a large number like 99 for INF (no connection):")
    matrix = []
    for i in range(n):
        while True:
            row = input(f"Row {i+1}: ").strip().split()
            if len(row) == n:
                try:
                    parsed = []
                    for j, x in enumerate(row):
                        val = float(x)
                        if i == j:
                            parsed.append(0)
                        elif val >= 99:
                            parsed.append(INF)
                        else:
                            parsed.append(val)
                    matrix.append(parsed)
                    break
                except ValueError:
                    pass
            print(f"Invalid. Enter exactly {n} numbers.")
    return matrix

def floyd_warshall(graph):
    n = len(graph)
    dist = copy.deepcopy(graph)
    for i in range(n):
        dist[i][i] = 0

    for k in range(n):
        for i in range(n):
            for j in range(n):
                if dist[i][k] + dist[k][j] < dist[i][j]:
                    dist[i][j] = dist[i][k] + dist[k][j]
    return dist

def detect_negative_cycle(graph):
    dist = floyd_warshall(graph)
    n = len(dist)
    for i in range(n):
        if dist[i][i] < 0:
            return True, i
    return False, -1

def reconstruct_path(graph, src, dst):
    n = len(graph)
    dist = copy.deepcopy(graph)
    for i in range(n):
        dist[i][i] = 0
    next_node = [[None] * n for _ in range(n)]

    for i in range(n):
        for j in range(n):
            if i != j and graph[i][j] < INF:
                next_node[i][j] = j

    for k in range(n):
        for i in range(n):
            for j in range(n):
                if dist[i][k] + dist[k][j] < dist[i][j]:
                    dist[i][j] = dist[i][k] + dist[k][j]
                    next_node[i][j] = next_node[i][k]

    if next_node[src][dst] is None:
        return None, INF

    path = [src]
    cur = src
    while cur != dst:
        cur = next_node[cur][dst]
        if cur is None:
            return None, INF
        path.append(cur)
    return path, dist[src][dst]

def dijkstra(graph, src):
    n = len(graph)
    dist = [INF] * n
    dist[src] = 0
    pq = [(0, src)]

    while pq:
        d, u = heapq.heappop(pq)
        if d > dist[u]:
            continue
        for v in range(n):
            if graph[u][v] < INF and u != v:
                nd = dist[u] + graph[u][v]
                if nd < dist[v]:
                    dist[v] = nd
                    heapq.heappush(pq, (nd, v))
    return dist

def print_matrix(matrix, label, n):
    print(f"\n{label}:")
    header = "     " + "  ".join(f"V{j+1:>3}" for j in range(n))
    print(header)
    print("     " + "-" * (6 * n))
    for i, row in enumerate(matrix):
        vals = []
        for v in row:
            vals.append("  INF" if v == INF else f"{v:>5.0f}")
        print(f"V{i+1:>2} |" + "".join(vals))

def print_path(path, cost, src, dst):
    if path is None:
        print(f"\n  No path from V{src+1} to V{dst+1}.")
    else:
        readable = " -> ".join(f"V{v+1}" for v in path)
        print(f"\n  Shortest path V{src+1} -> V{dst+1}: {readable}  (cost: {cost:.0f})")

def compare_with_dijkstra(graph, n):
    print("\nComparison — Floyd-Warshall vs Dijkstra (per source):")
    fw = floyd_warshall(graph)
    match = True
    for src in range(n):
        dijk = dijkstra(graph, src)
        for dst in range(n):
            fw_val = fw[src][dst] if fw[src][dst] < INF else INF
            dj_val = dijk[dst]
            if abs((fw_val or 0) - (dj_val or 0)) > 1e-9 and not (fw_val == INF and dj_val == INF):
                print(f"  Mismatch V{src+1}->V{dst+1}: Floyd={fw_val}, Dijkstra={dj_val}")
                match = False
    if match:
        print("  All distances match between Floyd-Warshall and Dijkstra.")

def main():
    n = int(input("Enter number of vertices: ").strip())

    graph = get_matrix_from_user(n)

    print_matrix(graph, "Input Matrix", n)
    has_neg_cycle, neg_node = detect_negative_cycle(graph)
    if has_neg_cycle:
        print(f"\nNegative cycle detected involving V{neg_node+1}! Floyd-Warshall results may be invalid.")
    else:
        print("\nNo negative cycle detected.")
    result = floyd_warshall(graph)
    print_matrix(result, "Shortest Distance Matrix (Floyd-Warshall)", n)

    print("\nFind shortest path between two vertices:")
    try:
        src = int(input(f"  Source vertex (1-{n}): ").strip()) - 1
        dst = int(input(f"  Destination vertex (1-{n}): ").strip()) - 1
        if 0 <= src < n and 0 <= dst < n:
            path, cost = reconstruct_path(graph, src, dst)
            print_path(path, cost, src, dst)
        else:
            print("  Invalid vertex numbers.")
    except ValueError:
        print("  Invalid input.")
    compare_with_dijkstra(graph, n)

if __name__ == "__main__":
    main()

In [ ]:
INF = float('inf')

graph = [
[0, 3, INF, 7],
[8, 0, 2, INF],
[5, INF, 0, 1],
[2, INF, INF, 0]
]

result = floyd_warshall(graph)

print("Shortest distance matrix:")

for row in result:
    print(row)

Shortest distance matrix:
[0, 3, 5, 6]
[5, 0, 2, 3]
[3, 6, 0, 1]
[2, 5, 7, 0]


# Excercise
1. Take matrix input from user
2. Detect negative cycle in graph
3. Print shortest path between two given vertices
4. Compare with Dijkstra algorithm